# Sección 9: Invariante 1 — Investigación del GUID Centinela

Este notebook investiga en profundidad el único violador de la Invariante 1:
el GUID centinela `∅` (`00000000-0000-0000-0000-000000000000`).

**Pregunta central:** ¿podemos recuperar el GUID correcto para los eventos
que Sysmon registró sin GUID real durante el boot?

**Archivos de entrada**
- `02_sysmon-run-01.csv` — CSV completo, ordenado cronológicamente
- `04_sysmon-run-01-violations.csv` — todos los eventos que participan en alguna violación
- `04_processguid-pid-violations-run-01.csv` — catálogo de violaciones Tipo 1

---

## Paso 1: Importaciones y constantes

In [ ]:
import pandas as pd
import numpy as np
import warnings

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 80)
pd.set_option('display.max_colwidth', 80)
warnings.filterwarnings('ignore')

NULL_GUID = '00000000-0000-0000-0000-000000000000'
WINDOW    = 15   # eventos antes y después en la ventana de contexto
DATA_DIR  = '../dataset/run-01-apt-1/'

print('OK')

## Paso 2: Carga de datos

In [ ]:
df = pd.read_csv(DATA_DIR + '02_sysmon-run-01.csv', low_memory=False)
df['_original_row_index'] = df.index
df['ts'] = pd.to_datetime(df['timestamp'].where(df['timestamp'] > 0), unit='ms')

print(f'df: {len(df):,} filas | {df.columns.size} columnas')
print(f'Rango temporal: {df["ts"].min()} → {df["ts"].max()}')

In [ ]:
viol = pd.read_csv(DATA_DIR + '04_sysmon-run-01-violations.csv', low_memory=False)
viol['ts'] = pd.to_datetime(viol['timestamp'].where(viol['timestamp'] > 0), unit='ms')

pid_viol = pd.read_csv(DATA_DIR + '04_processguid-pid-violations-run-01.csv')

print(f'viol    : {len(viol):,} eventos')
print(f'pid_viol: {len(pid_viol)} filas')

---
## Vista general de violaciones Tipo 1

La Invariante 1 establece que un ProcessGuid debe corresponder a exactamente un PID
en un mismo Computer. Verificamos cuántos GUIDs la violan y en qué k-pairs.

In [ ]:
summary = (
    pid_viol
    .groupby('k_pair')
    .agg(
        guids_violadores =('ProcessGuid', 'nunique'),
        pids_distintos   =('ProcessId',   'nunique'),
        filas_catalogo   =('ProcessGuid', 'count'),
    )
    .reset_index()
)
print('Resumen violaciones Tipo 1 por k-pair:')
display(summary)

print(f'\nGUIDs únicos violadores en total : {pid_viol["ProcessGuid"].nunique()}')
print(f'¿Es solo el centinela?           : {(pid_viol["ProcessGuid"] == NULL_GUID).all()}')

**Observación:** el único violador de la Invariante 1 es el GUID centinela `∅`
en los tres k-pairs donde aparece (k=1, k=2, k=4).
Todos los GUIDs reales tienen exactamente un PID → la Invariante 1 **sí se cumple**
para el universo de GUIDs reales generados por Sysmon.

---
## Invariante 1 en k=1: `ProcessGuid` / `ProcessId`

Dominio: eventos con EID ∉ {8, 10}.

In [ ]:
sentinel_k1 = (
    viol[
        ~viol['EventID'].isin([8, 10]) &
        (viol['ProcessGuid'] == NULL_GUID)
    ]
    .sort_values('_original_row_index')
    .reset_index(drop=True)
)

print(f'Eventos centinela k=1      : {len(sentinel_k1)}')
print(f'PIDs distintos             : {sentinel_k1["ProcessId"].nunique()}')
print(f'Computers distintos        : {sentinel_k1["Computer"].nunique()}')
print(f'Rango temporal             : {sentinel_k1["ts"].min()} → {sentinel_k1["ts"].max()}')
print(f'\nDistribución por EventID:')
print(sentinel_k1['EventID'].value_counts().sort_index())
print(f'\nDistribución por Computer:')
print(sentinel_k1['Computer'].value_counts())

In [ ]:
# Tabla completa: cada (ProcessId, Image, Computer) del centinela en k=1
cols = ['ts', 'EventID', 'ProcessId', 'Image', 'Computer']
sentinel_k1[cols]

### Ventana de contexto: primer evento centinela (k=1)

In [ ]:
first = sentinel_k1.iloc[0]
idx   = int(first['_original_row_index'])

window = df.iloc[max(0, idx - WINDOW) : idx + WINDOW + 1].copy()
window['▶'] = ''
window.loc[idx, '▶'] = '◀ centinela'

cols_w = ['▶', 'ts', 'EventID', 'ProcessGuid', 'ProcessId', 'Image', 'Computer']
window[cols_w].reset_index(drop=True)

---
## Invariante 1 en k=2: `ParentProcessGuid` / `ParentProcessId`

Dominio: eventos con EID = 1 (ProcessCreate).
El centinela aquí aparece como **padre** — son procesos lanzados durante el boot
cuyo padre Sysmon no pudo atribuir a ningún GUID real.

In [ ]:
sentinel_k2 = (
    viol[
        (viol['EventID'] == 1) &
        (viol['ParentProcessGuid'] == NULL_GUID)
    ]
    .sort_values('_original_row_index')
    .reset_index(drop=True)
)

print(f'Eventos centinela k=2      : {len(sentinel_k2)}')
print(f'PIDs hijo distintos (Image): {sentinel_k2["Image"].nunique()}')
print(f'PIDs padre distintos       : {sentinel_k2["ParentProcessId"].nunique()}')
print(f'Computers distintos        : {sentinel_k2["Computer"].nunique()}')
print(f'Rango temporal             : {sentinel_k2["ts"].min()} → {sentinel_k2["ts"].max()}')
print(f'\nTop 10 Images (procesos hijo lanzados desde centinela):')
print(sentinel_k2['Image'].value_counts().head(10))

In [ ]:
# Ventana de contexto del primer evento centinela en k=2
first_k2 = sentinel_k2.iloc[0]
idx_k2   = int(first_k2['_original_row_index'])

window_k2 = df.iloc[max(0, idx_k2 - WINDOW) : idx_k2 + WINDOW + 1].copy()
window_k2['▶'] = ''
window_k2.loc[idx_k2, '▶'] = '◀ centinela k=2'

cols_w = ['▶', 'ts', 'EventID', 'ProcessGuid', 'ProcessId',
          'ParentProcessGuid', 'ParentProcessId', 'Image', 'Computer']
window_k2[cols_w].reset_index(drop=True)

---
## Invariante 1 en k=4: `TargetProcessGUID` / `TargetProcessId`

Dominio: eventos con EID ∈ {8, 10} (CreateRemoteThread / ProcessAccess).
El centinela aparece como **proceso objetivo** de una inyección o acceso remoto.

In [ ]:
sentinel_k4 = (
    viol[
        viol['EventID'].isin([8, 10]) &
        (viol['TargetProcessGUID'] == NULL_GUID)
    ]
    .sort_values('_original_row_index')
    .reset_index(drop=True)
)

print(f'Eventos centinela k=4      : {len(sentinel_k4)}')
print(f'TargetProcessIds distintos : {sentinel_k4["TargetProcessId"].nunique()}')
print(f'Computers distintos        : {sentinel_k4["Computer"].nunique()}')
print(f'\nDetalle:')
cols_k4 = ['ts', 'EventID', 'TargetProcessGUID', 'TargetProcessId', 'TargetImage', 'Computer']
display(sentinel_k4[cols_k4])

In [ ]:
# Ventana de contexto del primer evento centinela en k=4
first_k4 = sentinel_k4.iloc[0]
idx_k4   = int(first_k4['_original_row_index'])

window_k4 = df.iloc[max(0, idx_k4 - WINDOW) : idx_k4 + WINDOW + 1].copy()
window_k4['▶'] = ''
window_k4.loc[idx_k4, '▶'] = '◀ centinela k=4'

cols_w = ['▶', 'ts', 'EventID', 'ProcessGuid', 'ProcessId',
          'TargetProcessGUID', 'TargetProcessId', 'TargetImage', 'Computer']
window_k4[[c for c in cols_w if c in window_k4.columns]].reset_index(drop=True)

---
## Investigación: ¿podemos recuperar el GUID correcto?

### Estrategia de recuperación para k=1

Cada evento centinela tiene `(Computer, ProcessId, timestamp)` conocidos.
El GUID correcto debería haber sido registrado en el **EID=1** (ProcessCreate)
del mismo proceso. Buscamos:

> Para cada evento centinela de k=1: ¿existe un EID=1 en el mismo
> `Computer` con el mismo `ProcessId` cuyo timestamp sea ≤ al del evento centinela?
> Si existe, tomamos el más reciente — ese es el candidato al GUID correcto.

Esta estrategia asume que el EID=1 del proceso fue registrado con GUID real
(lo cual es plausible si el artefacto centinela solo afecta ciertos tipos de eventos).

In [ ]:
# Tabla de EID=1 con GUID real: base de búsqueda
eid1 = (
    df[
        (df['EventID'] == 1) &
        (df['ProcessGuid'] != NULL_GUID) &
        df['ProcessGuid'].notna()
    ]
    [['Computer', 'ProcessId', 'ProcessGuid', 'Image', 'ts']]
    .copy()
)

print(f'EID=1 con GUID real disponibles: {len(eid1):,}')

In [ ]:
# Para cada evento centinela k=1, buscar el EID=1 más reciente
# con mismo Computer + ProcessId y timestamp ≤ sentinel timestamp

records = []
for _, row in sentinel_k1.iterrows():
    candidates = eid1[
        (eid1['Computer']   == row['Computer']) &
        (eid1['ProcessId']  == row['ProcessId']) &
        (eid1['ts']         <= row['ts'])
    ]
    if len(candidates) > 0:
        match = candidates.sort_values('ts').iloc[-1]  # más reciente
        records.append({
            'orig_idx'    : int(row['_original_row_index']),
            'Computer'    : row['Computer'],
            'ProcessId'   : row['ProcessId'],
            'sentinel_ts' : row['ts'],
            'sentinel_img': row['Image'],
            'candidate_guid' : match['ProcessGuid'],
            'candidate_img'  : match['Image'],
            'candidate_ts'   : match['ts'],
            'dt_seg'      : (row['ts'] - match['ts']).total_seconds(),
            'recovered'   : True,
        })
    else:
        records.append({
            'orig_idx'    : int(row['_original_row_index']),
            'Computer'    : row['Computer'],
            'ProcessId'   : row['ProcessId'],
            'sentinel_ts' : row['ts'],
            'sentinel_img': row['Image'],
            'candidate_guid' : None,
            'candidate_img'  : None,
            'candidate_ts'   : None,
            'dt_seg'      : None,
            'recovered'   : False,
        })

recovery = pd.DataFrame(records)

n_total     = len(recovery)
n_recovered = recovery['recovered'].sum()
n_lost      = (~recovery['recovered']).sum()

print(f'Eventos centinela k=1 : {n_total}')
print(f'  Con GUID candidato  : {n_recovered}  ({100*n_recovered/n_total:.0f}%)')
print(f'  Sin candidato (boot): {n_lost}  ({100*n_lost/n_total:.0f}%)')

In [ ]:
# Detalle completo: resultado de la búsqueda para cada evento centinela
cols_r = ['orig_idx', 'Computer', 'ProcessId', 'sentinel_ts', 'sentinel_img',
          'candidate_guid', 'candidate_img', 'dt_seg', 'recovered']
recovery[cols_r]

### Análisis de los resultados de recuperación

Para los eventos recuperados: ¿el `candidate_img` (Image del EID=1 encontrado)
coincide con el `sentinel_img` (Image del evento centinela)?
Esa coincidencia es evidencia de que la asignación es correcta.

In [ ]:
recovered = recovery[recovery['recovered']].copy()

if len(recovered) > 0:
    # Comparar Images (case-insensitive)
    recovered['img_match'] = (
        recovered['sentinel_img'].str.lower() == recovered['candidate_img'].str.lower()
    )
    # Para centinelas con '<unknown process>' la Image no es comparable
    recovered['img_comparable'] = recovered['sentinel_img'] != '<unknown process>'

    comparables = recovered[recovered['img_comparable']]
    print(f'Recuperados con Image comparable : {len(comparables)}')
    if len(comparables) > 0:
        print(f'  Image coincide               : {comparables["img_match"].sum()}')
        print(f'  Image NO coincide            : {(~comparables["img_match"]).sum()}')
        print()
        print('Casos donde Image NO coincide (revisar manualmente):')
        no_match = comparables[~comparables['img_match']]
        if len(no_match) > 0:
            display(no_match[['Computer','ProcessId','sentinel_img','candidate_img','dt_seg']])
        else:
            print('  Ninguno — todas las Images coinciden.')
else:
    print('No hay eventos recuperados.')

In [ ]:
# Distribución del delta temporal (segundos entre el EID=1 candidato y el evento centinela)
if len(recovered) > 0:
    print('Delta temporal (seg) entre EID=1 candidato y evento centinela:')
    print(recovered['dt_seg'].describe().round(2))
    print()
    print('Valores individuales:')
    print(recovered[['Computer','ProcessId','sentinel_img','dt_seg']].to_string(index=False))

### Eventos irrecuperables

Los eventos sin EID=1 candidato son artefactos de boot puros: Sysmon capturó
el evento pero el proceso arranció antes de que el driver tuviera visibilidad completa
y nunca generó un EID=1 con GUID real.

In [ ]:
lost = recovery[~recovery['recovered']]

if len(lost) > 0:
    print(f'Eventos irrecuperables: {len(lost)}')
    display(lost[['orig_idx','Computer','ProcessId','sentinel_ts','sentinel_img']])
else:
    print('Todos los eventos centinela k=1 tienen un EID=1 candidato.')

---
## Conclusión provisional: regla de corrección para k=1

A partir de la investigación anterior podemos formular la regla:

| Condición | Acción |
|-----------|--------|
| Evento centinela k=1 con EID=1 candidato (Image coincide) | Reemplazar `ProcessGuid = ∅` con `candidate_guid` |
| Evento centinela k=1 con EID=1 candidato (Image no coincide) | Marcar para revisión manual — posible reuso de PID |
| Evento centinela k=1 sin EID=1 candidato | Marcar como `BOOT_ARTIFACT` — excluir de cadenas causales |

Esta regla se implementará en el script `10_sysmon_data_cleaner.py`.

---
## Actividad Práctica

### Ejercicio A — Recuperación de GUID para k=2

El centinela en k=2 aparece como `ParentProcessGuid` en 500 eventos EID=1.
El GUID correcto del **padre** debería estar en otros eventos de ese padre
(por ejemplo, el propio EID=1 del padre, o cualquier evento EID ∉ {8,10}
con mismo `Computer` + `ParentProcessId` que tenga GUID real en `ProcessGuid`).

Implementa la estrategia de recuperación para k=2 y responde:
- ¿Qué fracción de los 500 eventos tiene un `ParentProcessGuid` recuperable?
- ¿Qué `ParentProcessId` valores son irrecuperables? ¿Cuál es su Image típica?

In [ ]:
# --- Ejercicio A ---
# Tabla de búsqueda: eventos con GUID real en ProcessGuid (EID ∉ {8,10})
# que puedan servir como referencia del proceso padre
real_guids = (
    df[
        ~df['EventID'].isin([8, 10]) &
        (df['ProcessGuid'] != NULL_GUID) &
        df['ProcessGuid'].notna()
    ]
    [['Computer', 'ProcessId', 'ProcessGuid', 'Image', 'ts']]
    .copy()
)

print(f'Eventos con GUID real disponibles para búsqueda: {len(real_guids):,}')

# TODO: para cada evento en sentinel_k2, buscar en real_guids por
#       Computer == sentinel_k2['Computer'] y
#       ProcessId == sentinel_k2['ParentProcessId'] y
#       ts <= sentinel_k2['ts']
# TODO: calcular tasa de recuperación
# TODO: listar ParentProcessId irrecuperables con su Image

### Ejercicio B — Estrategia de recuperación para k=4

Para los 2 eventos sentinela de k=4 (`TargetProcessGUID = ∅`),
busca el GUID correcto del proceso objetivo usando la misma estrategia
que en k=1 (EID=1 con mismo `Computer` + `TargetProcessId`).

- ¿Se pueden recuperar? ¿Qué proceso era el objetivo?
- ¿Qué tipo de evento (EID=8 o EID=10) los genera?
- ¿Qué proceso SOURCE realizó la operación sobre ese objetivo?

In [ ]:
# --- Ejercicio B ---
print('Detalle de los eventos centinela k=4:')
cols_b = ['ts', 'EventID', 'Computer', 'ProcessGuid', 'ProcessId', 'Image',
          'TargetProcessGUID', 'TargetProcessId', 'TargetImage']
display(sentinel_k4[[c for c in cols_b if c in sentinel_k4.columns]])

# TODO: buscar EID=1 con Computer == sentinel_k4['Computer']
#       y ProcessId == sentinel_k4['TargetProcessId']
#       y ts <= sentinel_k4['ts']
# TODO: mostrar el GUID candidato y validar contra TargetImage